In [1]:
import os
import zipfile
import pandas as pd
from pathlib import Path
from barley_analysis import analyze_barley_scan

In [ ]:
import pandas as pd
from pathlib import Path
from barley_analysis import analyze_barley_scan

# 每个 .tif 文件是一个独立样本，直接逐个调用 analyze_barley_scan
tiff_dir = Path("data/data3")
tiff_files = sorted(tiff_dir.glob("*.tif"))

if not tiff_files:
    raise FileNotFoundError(f"未找到 TIFF 文件: {tiff_dir}")

results = []
for tiff_file in tiff_files:
    try:
        print(f"Processing: {tiff_file.name}")
        result = analyze_barley_scan(str(tiff_file))
        results.append({
            "filename": tiff_file.name,
            "mean_wall_thickness_mm": result["mean_wall_thickness_mm"],
            "median_wall_thickness_mm": result["median_wall_thickness_mm"],
            "mean_per_mm_intensity": result["mean_per_mm_intensity"],
            "median_per_mm_intensity": result["median_per_mm_intensity"],
            "status": "success",
        })
    except Exception as e:
        print(f"Failed: {tiff_file.name} - {e}")
        results.append({
            "filename": tiff_file.name,
            "mean_wall_thickness_mm": None,
            "median_wall_thickness_mm": None,
            "mean_per_mm_intensity": None,
            "median_per_mm_intensity": None,
            "status": f"failed: {e}",
        })

result_test_tiff = pd.DataFrame(results)
print(f"\nProcessed {len(result_test_tiff)} files")
result_test_tiff

In [3]:
result_test_tiff.to_csv("result_test_tiff.csv", index=False)

In [25]:
from pathlib import Path

import phenoct
import tifffile

rek_dir = Path("data/0820samples")
out_dir = Path("data/0820samples_tif")
out_dir.mkdir(parents=True, exist_ok=True)

# ── 分割参数（可调）───────────────────────────────────────────────
START_SLICE = 750
STOP_SLICE = 2600
TUBE_R = 170
TUBE_THICKNESS = 30
ATTENUATION_THRESHOLD = 2000
# ────────────────────────────────────────────────────────────────

rek_files = sorted(rek_dir.glob("*.rek"))
if not rek_files:
    raise FileNotFoundError(f"No .rek files found in {rek_dir}")

converted = []
for rek_path in rek_files:
    print(f"Processing: {rek_path.name}")

    tube = phenoct.Tube(str(rek_path))
    tube.segment_sample_holder(
        start_slice=START_SLICE,
        stop_slice=STOP_SLICE,
        tube_r=TUBE_R,
        tube_thickness=TUBE_THICKNESS,
        attenuation_threshold=ATTENUATION_THRESHOLD,
        debug=False,
    )

    volume = tube.segmented_data
    out_path = out_dir / f"{rek_path.stem}.tif"
    tifffile.imwrite(str(out_path), volume)

    converted.append({
        "rek": rek_path.name,
        "tif": out_path.name,
        "shape": tuple(volume.shape),
        "dtype": str(volume.dtype),
    })

converted


Processing: X-049592.rek
uint16


Segmenting slice: 2599: 100%|██████████| 1850/1850 [00:12<00:00, 152.66it/s]


Processing: X-049593.rek
uint16


Segmenting slice: 2599: 100%|██████████| 1850/1850 [00:11<00:00, 154.88it/s]


Processing: X-049594.rek
uint16


Segmenting slice: 2599: 100%|██████████| 1850/1850 [00:11<00:00, 157.53it/s]


[{'rek': 'X-049592.rek',
  'tif': 'X-049592.tif',
  'shape': (2760, 404, 432),
  'dtype': 'uint16'},
 {'rek': 'X-049593.rek',
  'tif': 'X-049593.tif',
  'shape': (2760, 422, 404),
  'dtype': 'uint16'},
 {'rek': 'X-049594.rek',
  'tif': 'X-049594.tif',
  'shape': (2760, 422, 388),
  'dtype': 'uint16'}]

In [8]:
from barley_analysis import show_tiff_volume

# Pass only the TIFF filename/path to visualize in napari
tif_file = "data/0820samples_tif/X-049593.tif"
viewer, vol = show_tiff_volume(tif_file)
print(f"Viewing: {tif_file}")
print(f"shape={vol.shape}, dtype={vol.dtype}, min={vol.min()}, max={vol.max()}")

Viewing: data/0820samples_tif/X-049593.tif
shape=(2760, 422, 404), dtype=uint16, min=0, max=18077


/opt/anaconda3/envs/napari-stable/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2760, 422, 404) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


In [ ]:
from barley_analysis import show_tiff_volume
viewer, vol = show_tiff_volume('data/0820/X-050856.tif')

/opt/anaconda3/envs/napari-stable/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2760, 391, 430) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


: 

In [9]:
from barley_analysis import analyze_barley_scan
analyze_barley_scan('data/0820samples_tif/X-049593.tif')

z1:1414, z2:2010


{'mean_wall_thickness_mm': 0.2934298411532657,
 'median_wall_thickness_mm': 0.2910876309303147,
 'mean_per_mm_intensity': 2424.315414776381,
 'median_per_mm_intensity': 2406.5546218487393}

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

# ===== Config =====
batch_dirs = {
    "0812": Path("data/0812"),
    "0820": Path("data/0820"),
}

slice_ranges = [
    (0, 500),
    (500, 1000),
    (1000, 1500),
    (1500, 2000),
]

records = []
for batch, folder in batch_dirs.items():
    tif_files = sorted(folder.glob("*.tif"))
    if not tif_files:
        raise FileNotFoundError(f"No TIFF files found in {folder}")

    print(f"Batch {batch}: {len(tif_files)} files")
    for i, tif_path in enumerate(tif_files, start=1):
        vol = tifffile.imread(str(tif_path))
        if vol.ndim != 3:
            raise ValueError(f"Expected 3D TIFF, got {vol.shape} in {tif_path.name}")

        # Whole-sample mean intensity (mean of per-slice means)
        sample_mean_intensity = float(vol.mean(axis=(1, 2)).mean())

        row = {
            "file": tif_path.name,
            "sample_mean_intensity": sample_mean_intensity,
        }

        # Range-wise means (left-closed, right-open; e.g., 0-500 => [0, 500))
        for start, stop in slice_ranges:
            range_start = max(0, start)
            range_stop = min(vol.shape[0], stop)
            col_name = f"mean_intensity_{start}_{stop}"

            if range_start >= range_stop:
                row[col_name] = np.nan
            else:
                row[col_name] = float(vol[range_start:range_stop].mean())

        records.append((batch, row))

        if i % 100 == 0 or i == len(tif_files):
            print(f"  processed {i}/{len(tif_files)} files in batch {batch}")

# Split into two batches and save (keep only requested columns)
target_cols = [
    "file",
    "sample_mean_intensity",
    "mean_intensity_0_500",
    "mean_intensity_500_1000",
    "mean_intensity_1000_1500",
    "mean_intensity_1500_2000",
]

df_0812 = pd.DataFrame([r for b, r in records if b == "0812"])[target_cols].sort_values("file").reset_index(drop=True)
df_0820 = pd.DataFrame([r for b, r in records if b == "0820"])[target_cols].sort_values("file").reset_index(drop=True)

out_0812 = "sample_mean_intensity_0812.csv"
out_0820 = "sample_mean_intensity_0820.csv"

df_0812.to_csv(out_0812, index=False)
df_0820.to_csv(out_0820, index=False)

print("Saved:")
print(f"- {out_0812}: {len(df_0812)} rows")
print(f"- {out_0820}: {len(df_0820)} rows")

display(df_0812.head())
display(df_0820.head())

Batch 0812: 900 files
  processed 100/900 files in batch 0812
  processed 200/900 files in batch 0812
  processed 300/900 files in batch 0812
  processed 400/900 files in batch 0812
  processed 500/900 files in batch 0812
  processed 600/900 files in batch 0812
  processed 700/900 files in batch 0812
  processed 800/900 files in batch 0812
  processed 900/900 files in batch 0812
Batch 0820: 900 files
  processed 100/900 files in batch 0820
  processed 200/900 files in batch 0820
  processed 300/900 files in batch 0820
  processed 400/900 files in batch 0820
  processed 500/900 files in batch 0820
  processed 600/900 files in batch 0820
  processed 700/900 files in batch 0820
  processed 800/900 files in batch 0820
  processed 900/900 files in batch 0820
Saved:
- sample_mean_intensity_0812.csv: 900 rows
- sample_mean_intensity_0820.csv: 900 rows


,file,sample_mean_intensity,mean_intensity_0_500,mean_intensity_500_1000,mean_intensity_1000_1500,mean_intensity_1500_2000
0,X-040501.tif,46.516076,0.0,94.940326,147.409809,1.655382
1,X-040502.tif,44.941780,0.0,100.405461,128.728124,3.418135
2,X-040503.tif,56.841535,0.0,117.078425,175.665475,3.663872
3,X-040504.tif,50.512313,0.0,68.482242,190.907627,3.854103
4,X-040505.tif,57.133210,0.0,107.613444,190.107041,3.060973


,file,sample_mean_intensity,mean_intensity_0_500,mean_intensity_500_1000,mean_intensity_1000_1500,mean_intensity_1500_2000
0,X-049501.tif,35.077516,0.0,31.606492,151.646808,0.008934
1,X-049502.tif,44.632674,0.0,79.319858,153.040303,1.272519
2,X-049503.tif,47.834658,0.0,91.216696,157.439411,1.700285
3,X-049504.tif,50.032506,0.0,103.037226,158.989777,1.835847
4,X-049505.tif,33.962488,0.0,64.521809,111.818211,0.100234
